## frame_fraction.py
Create a grid of SAXS parameters and send a slurm job script to simulate structures through Pepsi SAXS

Python module imports 

Define paths to where structure files are

Create an output folder with date and run number for orgnization purposes

In [ ]:
import os
import subprocess
import sys
import pandas as pd
import numpy as np
from datetime import date
import time
import glob
import re

#Date Time
epoch = time.time()
cd = time.strftime("%Y%m%d-%H%M%S", time.localtime(epoch))
today = date.today()

############---------- PATHS
path_frames = "/path/to/structure/files
folder_pattern = os.path.join(path_frames, "mm*", "mm*_*")
structure_folders = glob.glob(folder_pattern)

path_exp_file = "/users/t/j/tjaglal/experimental_data"
path_main = "/users/t/j/tjaglal/Projects/iBME"
path_shell = "/users/t/j/tjaglal/shell"

#Make output file
path_results = "/gpfs1/home/t/j/tjaglal/Projects/iBME/pepsi_result"
sub_dir_num = 0
run_fol = "run_{}_{}"
sub_path = os.path.join(path_results, run_fol)
while os.path.isdir(sub_path.format(today, sub_dir_num)):
    sub_dir_num += 1
sub_path = sub_path.format(today, sub_dir_num)
os.mkdir(sub_path)

Create grid function where the minimum, maximum, and step values of drho and r0 will be defined

In [ ]:
def create_grid(dro_min, dro_max, dro_step, r0_min, r0_max, r0_step):
    dro_grid = np.arange(dro_min, dro_max + dro_step, dro_step)
    r0_grid = np.arange(r0_min, r0_max + r0_step, r0_step)

    grid_data = []

    index = 0
    for d in dro_grid:
        dro_val = round(d, 2)
        for r in r0_grid:
            r0_val = round(r, 2)
            grid_data.append([index, dro_val, r0_val])
            index += 1

    grid_name = np.array(grid_data)

    return grid_name, index

Main script

Insert grid parameters and ensure that structure folders exist

Send job script to slurm as fractions. Will not simulate all structures for one grid point run. Instead, create n jobs for 100 structure fractions, and simulate every grid point within that fraction.

In [ ]:
theta_val = 100
gn, idx = create_grid(30, 60, 5, 1.35, 1.65, .05)
gdf = pd.DataFrame(gn, columns=['index', 'dro', 'r0'])

os.chdir(path_shell)

#Run Pepsi for fraction
fraction_job_ids = []

for struct_path in structure_folders:
    if os.path.isdir(struct_path):

        folder_name = os.path.basename(struct_path)
        spec_save = os.path.join(sub_path, folder_name)
        os.makedirs(spec_save, exist_ok=True)

        np.savetxt(
            os.path.join(spec_save, "grid_full.txt"),
            gn,
            fmt=['%d', '%.2f', '%.2f'])

        cmd = ["sbatch", "--parsable", "iBME_scan.sh", str(theta_val), spec_save, struct_path]
        run = subprocess.run(cmd, capture_output=True, text=True)

        if run.returncode == 0:
            job_id = run.stdout.strip()
            fraction_job_ids.append(job_id)
        else:
            print("Failed folder submit")

if fraction_job_ids:
    dependency_string = f"afterok:{':'.join(fraction_job_ids)}"
    
    print(f"\nSubmitting final iBME reweighting job with dependency on {len(fraction_job_ids)} jobs...")
    
    final_cmd = [
        "sbatch", 
        f"--dependency={dependency_string}", 
        "iBME_reweight.sh", 
        str(theta_val), 
        sub_path
    ]
    
    final_run = subprocess.run(final_cmd, capture_output=True, text=True)
    
    if final_run.returncode == 0:
        print(f"Success! Final job submitted: {final_run.stdout.strip()}")
    else:
        print(f"Failed to submit final job: {final_run.stderr}")
else:
    print("No fraction jobs were successfully submitted. Aborting final job.")